# NCAA D1 Baseball — 2026 Tournament Elo & Simulation (Python port)

Python rewrite of the 2025 R model, with the goal of dropping the R dependency entirely. Same algorithm as the 2025 R notebook:

1. Load cached 2021–2025 schedules + (when ready) the freshly-scraped 2026 schedules.
2. Clean to one row per game with W/L outcome.
3. Fit Elo from scratch (no `elo` R package equivalent — just plain Python).
4. Grid-search `k`, `hfa`, and a yearly regression-to-mean factor `r`.
5. Simulate the 2026 bracket: 16 regionals → super regionals → CWS — same single-game style as 2025.

**Improvements are intentionally NOT included.** This notebook reproduces last year's logic. We'll add MOV / recency / etc. only after the 2025 retrocast tells us they help.

In [1]:
from __future__ import annotations

import math
import os
from collections import defaultdict
from dataclasses import dataclass
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Resolve relative to where this notebook lives (pipeline/)
import os
PROJECT_ROOT = Path(os.getcwd())  # run the notebook from pipeline/
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

## 1. Load & combine 2021–2026 schedules

In [2]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from wn_to_ncaa_name_map import to_ncaa

hist = pd.read_csv(PROJECT_ROOT / 'd1_sched_raw.csv')
print(f"Cached 2021-2025 rows: {len(hist):,}")
print("Year breakdown:", hist['year'].value_counts().sort_index().to_dict())

# 2026 from Warren Nolan (stats.ncaa.org is Akamai-blocked since the 2025 run)
wn_path = PROJECT_ROOT / 'd1_sched_2026_wn_raw.csv'
if wn_path.exists():
    wn = pd.read_csv(wn_path)
    # Apply WN -> NCAA canonical name map so 2026 joins with cached schema
    wn['team_name'] = wn['team_name'].map(to_ncaa)
    wn['home_team'] = wn['home_team'].map(to_ncaa)
    wn['away_team'] = wn['away_team'].map(to_ncaa)
    print(f"\n2026 WN rows: {len(wn):,}")
else:
    wn = None
    print("\n[!] 2026 WN scrape not present. Run scrape_warrennolan_2026.py first.")

Cached 2021-2025 rows: 94,238
Year breakdown: {2021: 18193, 2022: 19523, 2023: 19717, 2024: 19703, 2025: 17102}

2026 WN rows: 16,594


## 2. Align 2026 (Warren Nolan) to cached schema and concat

stats.ncaa.org has been Akamai-blocked since the 2025 R run, so 2026 schedules
come from warrennolan.com via `scrape_warrennolan_2026.py`. The WN scrape
already produces clean home_team / away_team / scores / W-L outcome columns —
we just need to fill in the metadata columns (`team_id`, `conference`, etc.)
that the cached 21-25 file carries.

Name reconciliation lives in `wn_to_ncaa_name_map.py` — 86 WN team names map
to canonical stats.ncaa.org names. Only `New Haven` (recent D1 addition) has
no historical match; it'll start the season at 1500.

In [3]:
# Align WN 2026 to the cached d1_sched_raw schema, then concat.
# The WN scrape already produces clean home/away/score/result columns —
# we just need to fill in the metadata columns the cached file has.

if wn is not None:
    target_cols = list(hist.columns)
    for col in target_cols:
        if col not in wn.columns:
            wn[col] = pd.NA
    wn['team_id']       = pd.NA
    wn['conference']    = pd.NA
    wn['conference_id'] = pd.NA
    wn['division']      = 1
    wn['season_id']     = 'WN2026'
    wn['original_opponent'] = wn['team_slug']
    wn['original_result']   = (wn['game_result'].astype(str) + ' '
                              + wn['home_score'].astype(str) + '-'
                              + wn['away_score'].astype(str))
    wn_aligned = wn[target_cols].copy()
    combined = pd.concat([hist, wn_aligned], ignore_index=True)
else:
    combined = hist.copy()

print(f"Combined rows: {len(combined):,}")
print(f"Years: {sorted(combined['year'].dropna().unique().tolist())}")
combined.to_csv(PROJECT_ROOT / 'd1_sched_2126_combined.csv', index=False)

Combined rows: 110,832
Years: [2021, 2022, 2023, 2024, 2025, 2026]


/tmp/ipykernel_4006617/1495172877.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([hist, wn_aligned], ignore_index=True)


## 3. Game-level cleaning (one row per game)

The raw data has each game from both teams' perspectives. Slice to one row each.

In [4]:
games = combined[combined['game_result'].isin(['W','L'])].copy()

# Drop 2021 Ivy League (no/partial season)
games = games[~((games['year'] == 2021) & (games['conference'] == 'Ivy League'))]

# Match R's paste0 on numeric columns: integer-valued floats render without trailing .0.
# pandas' astype(str) emits '5.0', which causes dedup misses
# (R yields 43,246 unique games on cached 21-25; astype(str) only 34,389).
def _r_str(x):
    if pd.isna(x): return ''
    if isinstance(x, float) and x.is_integer(): return str(int(x))
    return str(x)

games['unique_id'] = (games['home_team'].apply(_r_str) + '_' +
                     games['away_team'].apply(_r_str) + '_' +
                     games['home_score'].apply(_r_str) + '_' +
                     games['away_score'].apply(_r_str) + '_' +
                     games['doubleheader_game'].apply(_r_str) + '_' +
                     games['season_id'].apply(_r_str))

games = games.drop_duplicates('unique_id', keep='first')
games['home_team_win'] = (games['home_score'].astype(float) > games['away_score'].astype(float)).astype(int)
games['Date'] = pd.to_datetime(games['Date'], format='%m/%d/%Y', errors='coerce')
games = games.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

print(f"Unique games for fitting: {len(games):,}")
print(f"Years: {sorted(games['year'].unique().tolist())}")

Unique games for fitting: 52,085
Years: [2021, 2022, 2023, 2024, 2025, 2026]


## 4. Elo from scratch

Equivalent to the 2025 R `elo.run` call:
- Expected win prob: `E_A = 1 / (1 + 10^((R_B - R_A) / 400))`
- Home-field advantage `h`: added to the home team's rating when computing E (skipped on `neutral_site == 1`).
- Update on game played: `R_A ← R_A + k * (actual - E_A)`, opposite for B.
- Yearly regression: at each year change, every team's rating shrinks toward its starting Elo by factor `r` (R's `regress(year, starting_elos, by = r)` does exactly this between consecutive year blocks).
- Returns final ratings + pre-game win prob for every game (for MSE/AUC scoring).

In [5]:
def expected_win_prob(rating_a: float, rating_b: float, hfa: float, neutral: int) -> float:
    """P(team A wins) given Elo ratings, hfa for A (0 on neutral), and pure Elo logistic."""
    adj_a = rating_a + (0.0 if neutral else hfa)
    diff = rating_b - adj_a
    return 1.0 / (1.0 + 10.0 ** (diff / 400.0))


@dataclass
class EloFitResult:
    final_ratings: dict[str, float]
    pregame_probs: np.ndarray  # P(home wins) before each game
    actuals: np.ndarray        # 1 if home won

    @property
    def mse(self) -> float:
        return float(np.mean((self.pregame_probs - self.actuals) ** 2))

    @property
    def auc(self) -> float:
        return float(roc_auc_score(self.actuals, self.pregame_probs))


def fit_elo(games: pd.DataFrame, starting: dict[str, float],
            k: float, hfa: float, regress_by: float) -> EloFitResult:
    ratings = dict(starting)  # mutable copy
    starting_for_regress = dict(starting)
    n = len(games)
    probs = np.empty(n)
    actuals = games['home_team_win'].to_numpy()

    home_arr = games['home_team'].to_numpy()
    away_arr = games['away_team'].to_numpy()
    neutral_arr = games['neutral_site'].to_numpy()
    year_arr = games['year'].to_numpy()

    prev_year = year_arr[0]
    for i in range(n):
        year = year_arr[i]
        if year != prev_year:
            # Regression to starting Elos between seasons
            for t in ratings:
                base = starting_for_regress.get(t, 1500.0)
                ratings[t] = ratings[t] + regress_by * (base - ratings[t])
            prev_year = year

        home, away = home_arr[i], away_arr[i]
        r_h = ratings.setdefault(home, 1500.0)
        r_a = ratings.setdefault(away, 1500.0)
        p_home = expected_win_prob(r_h, r_a, hfa, neutral_arr[i])
        probs[i] = p_home

        outcome = actuals[i]
        ratings[home] = r_h + k * (outcome - p_home)
        ratings[away] = r_a + k * ((1 - outcome) - (1 - p_home))

    return EloFitResult(final_ratings=ratings, pregame_probs=probs, actuals=actuals)


### 4a. Sanity check: reproduce the R fit on cached 2021–2025

R 2025 cached optimum: `k=29, h=40, r=0.07`, MSE=0.2153, AUC=0.6957. Python should land within rounding.

In [6]:
all_team_names = sorted(set(games['home_team']) | set(games['away_team']))
starting_elos = {t: 1500.0 for t in all_team_names}

fit_cached = fit_elo(games, starting_elos, k=29, hfa=40, regress_by=0.07)
print(f"k=29 h=40 r=0.07  MSE={fit_cached.mse:.4f}  AUC={fit_cached.auc:.4f}")

top10 = sorted(fit_cached.final_ratings.items(), key=lambda kv: -kv[1])[:10]
print("\nTop 10 (Python):")
for nm, r in top10:
    print(f"  {nm:25s} {r:7.1f}")

k=29 h=40 r=0.07  MSE=0.2148  AUC=0.6982

Top 10 (Python):
  Georgia Tech               1972.9
  Georgia                    1971.6
  UCLA                       1943.7
  North Carolina             1901.9
  Arkansas                   1872.6
  Kansas                     1871.8
  Auburn                     1867.3
  Texas                      1850.0
  Florida                    1849.2
  West Virginia              1829.1


## 5. Grid search

Same grid as the 2025 R script: `k∈[10,30]`, `hfa∈[30,40]`, `r∈[0.05,0.20]`. ~3500 combinations × ~100k games each — ~5–15 minutes in Python depending on hardware. Pure NumPy doesn't help much because the inner loop is sequential (each game depends on the previous Elo), but the inner update is tight Python.

In [7]:
RUN_GRID_SEARCH = False  # Set True to re-search; takes time.

if RUN_GRID_SEARCH:
    k_options = range(10, 31)
    h_options = range(30, 41)
    r_options = [round(x, 2) for x in np.arange(0.05, 0.21, 0.01)]
    rows = []
    grid = list(product(k_options, h_options, r_options))
    for idx, (k, h, r) in enumerate(grid):
        if idx % 100 == 0:
            print(f"{idx}/{len(grid)}")
        res = fit_elo(games, starting_elos, k=k, hfa=h, regress_by=r)
        rows.append({'k': k, 'h': h, 'r': r, 'mse': res.mse, 'auc': res.auc})
    grid_df = pd.DataFrame(rows).sort_values('mse').reset_index(drop=True)
    grid_df.to_csv(PROJECT_ROOT / 'optimal_elo_params_2026_py.csv', index=False)
    print(grid_df.head())
    optim = grid_df.iloc[0]
    OPTIM_K, OPTIM_H, OPTIM_R = optim.k, optim.h, optim.r
else:
    OPTIM_K, OPTIM_H, OPTIM_R = 29, 40, 0.07
    print(f"Using cached params: k={OPTIM_K} h={OPTIM_H} r={OPTIM_R}")

Using cached params: k=29 h=40 r=0.07


In [8]:
fit = fit_elo(games, starting_elos, k=OPTIM_K, hfa=OPTIM_H, regress_by=OPTIM_R)
team_ranks = pd.DataFrame(
    sorted(fit.final_ratings.items(), key=lambda kv: -kv[1]),
    columns=['school', 'elo'])
team_ranks['rank'] = range(1, len(team_ranks) + 1)
print(f"Final fit: MSE={fit.mse:.4f}  AUC={fit.auc:.4f}")
team_ranks.head(20)

Final fit: MSE=0.2148  AUC=0.6982


,school,elo,rank
0,Georgia Tech,1972.914500,1
1,Georgia,1971.605918,2
2,UCLA,1943.725809,3
3,North Carolina,1901.861018,4
4,Arkansas,1872.629519,5
5,Kansas,1871.759346,6
6,Auburn,1867.336813,7
7,Texas,1849.959815,8
8,Florida,1849.249868,9
9,West Virginia,1829.082704,10


## 6. 2026 bracket

Pulled from ncaa.com/brackets/baseball/d1/2026 (Selection Monday: 2026-05-25). Super-regional pairings follow the standard 1v16, 2v15, ..., 8v9 by national seed (officially announced 6/2 but fixed by the bracket math).

In [9]:
# bracket display name -> baseballr/stats.ncaa.org team_name
BRACKET_NAME_MAP = {
    "Saint Mary's CA": "Saint Mary's (CA)",
    "Miami FL":         "Miami (FL)",
    "St. John's NY":    "St. John's (NY)",
    "Tarleton State":   "Tarleton St.",
    "Oklahoma State":   "Oklahoma St.",
    "Alabama State":    "Alabama St.",
    "Southern Mississippi": "Southern Miss.",
    "Jacksonville State": "Jacksonville St.",
    "Florida State":    "Florida St.",
    "Oregon State":     "Oregon St.",
    "Washington State": "Washington St.",
    "Texas State":      "Texas St.",
    "Lamar University": "Lamar University",
    "Arizona State":    "Arizona St.",
    "South Dakota State": "South Dakota St.",
    "Mississippi State":"Mississippi St.",
    "Missouri State":   "Missouri St.",
    "Northern Illinois": "NIU",
}

def resolve(name: str) -> str:
    return BRACKET_NAME_MAP.get(name, name)

REGIONALS_2026 = {
    'los_angeles':     dict(seed=1,  host='UCLA',
                            teams=['UCLA','Virginia Tech','Cal Poly',"Saint Mary's CA"]),
    'atlanta':         dict(seed=2,  host='Georgia Tech',
                            teams=['Georgia Tech','Oklahoma','The Citadel','UIC']),
    'athens':          dict(seed=3,  host='Georgia',
                            teams=['Georgia','Boston College','Liberty','LIU']),
    'auburn':          dict(seed=4,  host='Auburn',
                            teams=['Auburn','UCF','NC State','Milwaukee']),
    'chapel_hill':     dict(seed=5,  host='North Carolina',
                            teams=['North Carolina','Tennessee','East Carolina','VCU']),
    'austin':          dict(seed=6,  host='Texas',
                            teams=['Texas','UC Santa Barbara','Tarleton State','Holy Cross']),
    'tuscaloosa':      dict(seed=7,  host='Alabama',
                            teams=['Alabama','Oklahoma State','USC Upstate','Alabama State']),
    'gainesville':     dict(seed=8,  host='Florida',
                            teams=['Florida','Miami FL','Troy','Rider']),
    'hattiesburg':     dict(seed=9,  host='Southern Mississippi',
                            teams=['Southern Mississippi','Virginia','Jacksonville State','Little Rock']),
    'tallahassee':     dict(seed=10, host='Florida State',
                            teams=['Florida State','Coastal Carolina','Northern Illinois',"St. John's NY"]),
    'eugene':          dict(seed=11, host='Oregon',
                            teams=['Oregon','Oregon State','Washington State','Yale']),
    'college_station': dict(seed=12, host='Texas A&M',
                            teams=['Texas A&M','Southern California','Texas State','Lamar University']),
    'lincoln':         dict(seed=13, host='Nebraska',
                            teams=['Nebraska','Ole Miss','Arizona State','South Dakota State']),
    'starkville':      dict(seed=14, host='Mississippi State',
                            teams=['Mississippi State','Cincinnati','Louisiana','Lipscomb']),
    'lawrence':        dict(seed=15, host='Kansas',
                            teams=['Kansas','Arkansas','Missouri State','Northeastern']),
    'morgantown':      dict(seed=16, host='West Virginia',
                            teams=['West Virginia','Wake Forest','Kentucky','Binghamton']),
}

# Map all to baseballr names
for name, r in REGIONALS_2026.items():
    r['host'] = resolve(r['host'])
    r['teams'] = [resolve(t) for t in r['teams']]

# Sanity check
all_bracket_teams = {t for r in REGIONALS_2026.values() for t in r['teams']}
missing = all_bracket_teams - set(team_ranks['school'])
if missing:
    print(f"[!] {len(missing)} bracket team(s) not in team_ranks (will use Elo=1500):")
    for m in sorted(missing):
        print(f"    {m}")
else:
    print(f"All {len(all_bracket_teams)} bracket teams matched team_ranks. ✓")

All 64 bracket teams matched team_ranks. ✓


In [10]:
# Super-regional pairings: 1v16, 2v15, ..., 8v9 by national seed
seed_to_regional = {r['seed']: name for name, r in REGIONALS_2026.items()}
SUPER_PAIRS = [
    (seed_to_regional[i], seed_to_regional[17 - i]) for i in range(1, 9)
]
for i, p in enumerate(SUPER_PAIRS, 1):
    print(f"Super {i}: {p[0]} vs {p[1]}")

Super 1: los_angeles vs morgantown
Super 2: atlanta vs lawrence
Super 3: athens vs starkville
Super 4: auburn vs lincoln
Super 5: chapel_hill vs college_station
Super 6: austin vs eugene
Super 7: tuscaloosa vs tallahassee
Super 8: gainesville vs hattiesburg


## 7. Simulation engine (2025-style baseline)

Same shape as the 2025 R simulator:
- Regional: 4-team double elimination, all games neutral (no HFA in 2025 R sim).
- Super regional: simulated as a single neutral game (not best-of-3).
- CWS: two 4-team double-elim brackets, then single championship game.

Will revisit HFA + best-of-3 + correct CWS bracketing AFTER the 2025 retrocast says weighting helps.

In [11]:
RATINGS = dict(zip(team_ranks['school'], team_ranks['elo']))

def get_elo(team: str) -> float:
    return RATINGS.get(team, 1500.0)

def sim_game(team1: str, team2: str, rng=rng) -> str:
    p1 = expected_win_prob(get_elo(team1), get_elo(team2), hfa=0.0, neutral=1)
    return team1 if rng.random() < p1 else team2

def sim_regional(teams: list[str], rng=rng) -> str:
    # 1v4, 2v3 → winners bracket; double elim
    g1 = sim_game(teams[0], teams[3], rng)
    g2 = sim_game(teams[1], teams[2], rng)
    g1l = teams[3] if g1 == teams[0] else teams[0]
    g2l = teams[2] if g2 == teams[1] else teams[1]
    wbf = sim_game(g1, g2, rng)
    wbfl = g2 if wbf == g1 else g1
    lg1 = sim_game(g1l, g2l, rng)
    lbf = sim_game(wbfl, lg1, rng)
    f1 = sim_game(wbf, lbf, rng)
    if f1 == lbf:
        return sim_game(wbf, lbf, rng)  # deciding game (wbf still has 0 losses)
    return f1

def sim_super(regional_winners: dict[str, str], rng=rng) -> list[str]:
    cws = []
    for r1, r2 in SUPER_PAIRS:
        cws.append(sim_game(regional_winners[r1], regional_winners[r2], rng))
    return cws

def sim_cws(cws_teams: list[str], rng=rng) -> str:
    # 2025 R style: simulate each 4-team bracket via the same double-elim engine
    b1 = sim_regional(cws_teams[:4], rng)
    b2 = sim_regional(cws_teams[4:], rng)
    return sim_game(b1, b2, rng)

def sim_tournament(rng=rng) -> dict:
    reg_winners = {nm: sim_regional(r['teams'], rng) for nm, r in REGIONALS_2026.items()}
    cws_teams = sim_super(reg_winners, rng)
    champion = sim_cws(cws_teams, rng)
    return dict(regional_winners=reg_winners, cws_teams=cws_teams, champion=champion)

# Smoke test — one sim
demo = sim_tournament()
print("Demo champion:", demo['champion'])
print("Demo CWS:", demo['cws_teams'])

Demo champion: Arizona St.
Demo CWS: ['Wake Forest', 'Kansas', 'Georgia', 'Arizona St.', 'North Carolina', 'Texas', 'Florida St.', 'Jacksonville St.']


## 8. Run many simulations

In [12]:
N_SIMS = 5000

champ_counts = defaultdict(int)
cws_counts = defaultdict(int)
regional_counts = {nm: defaultdict(int) for nm in REGIONALS_2026}

for i in range(N_SIMS):
    if (i + 1) % 500 == 0:
        print(f"  sim {i+1}/{N_SIMS}")
    s = sim_tournament()
    champ_counts[s['champion']] += 1
    for t in s['cws_teams']:
        cws_counts[t] += 1
    for nm, w in s['regional_winners'].items():
        regional_counts[nm][w] += 1

champ_probs = (pd.DataFrame(
    [(t, c / N_SIMS * 100) for t, c in champ_counts.items()],
    columns=['team', 'win_pct'])
    .sort_values('win_pct', ascending=False).reset_index(drop=True))
champ_probs['rank'] = champ_probs.index + 1
champ_probs = champ_probs[['rank', 'team', 'win_pct']]
champ_probs.head(15)

  sim 500/5000


  sim 1000/5000


  sim 1500/5000


  sim 2000/5000
  sim 2500/5000


  sim 3000/5000


  sim 3500/5000


  sim 4000/5000


  sim 4500/5000


  sim 5000/5000


,rank,team,win_pct
0,1,Georgia,18.92
1,2,Georgia Tech,15.96
2,3,UCLA,14.18
3,4,North Carolina,8.88
4,5,Texas,6.10
5,6,Auburn,5.10
6,7,Florida,4.60
7,8,Oregon,2.56
8,9,Texas A&M,2.42
9,10,Kansas,2.28


In [13]:
cws_probs = (pd.DataFrame(
    [(t, c / N_SIMS * 100) for t, c in cws_counts.items()],
    columns=['team', 'cws_pct'])
    .sort_values('cws_pct', ascending=False).reset_index(drop=True))
cws_probs['rank'] = cws_probs.index + 1
cws_probs = cws_probs[['rank', 'team', 'cws_pct']]
cws_probs.head(20)

,rank,team,cws_pct
0,1,Georgia,66.34
1,2,UCLA,59.56
2,3,Georgia Tech,56.48
3,4,Auburn,50.00
4,5,North Carolina,43.74
5,6,Texas,42.84
6,7,Florida,35.78
7,8,Alabama,33.56
8,9,Oregon,26.56
9,10,Texas A&M,25.14


In [14]:
print("Most-likely regional winners:")
for nm, ctr in regional_counts.items():
    top, ct = max(ctr.items(), key=lambda kv: kv[1])
    print(f"  {nm:18s} {top:30s} {ct / N_SIMS * 100:5.1f}%")

Most-likely regional winners:
  los_angeles        UCLA                            83.9%
  atlanta            Georgia Tech                    85.2%
  athens             Georgia                         88.2%
  auburn             Auburn                          76.6%
  chapel_hill        North Carolina                  64.7%
  austin             Texas                           73.2%
  tuscaloosa         Alabama                         53.9%
  gainesville        Florida                         62.8%
  hattiesburg        Southern Miss.                  43.0%
  tallahassee        Florida St.                     49.7%
  eugene             Oregon                          51.8%
  college_station    Texas A&M                       54.6%
  lincoln            Nebraska                        52.3%
  starkville         Mississippi St.                 58.5%
  lawrence           Kansas                          45.0%
  morgantown         West Virginia                   59.5%


## 11. 2025 retrocast — does weighting help?

Strategy: fit Elo on cached 21-25 data only (which cuts off at Selection Sunday 2025),
simulate the 2025 bracket with three Elo variants, compare predictions to actual
2025 tournament outcomes.

**Variants:**
1. **baseline** — same as 2025 R model (constant k=29).
2. **MOV-weighted** — per-game k modulated by FiveThirtyEight margin-of-victory multiplier.
3. **recency-weighted** — per-game k scaled by `exp(-days_before_cutoff / 60)`.

**Ground truth:**
- 2025 champion: **LSU**
- 2025 runner-up: **Coastal Carolina**
- 2025 CWS participants: Coastal Carolina, Arizona, Oregon St., Louisville, UCLA, Murray St., Arkansas, LSU.

**Metrics:** champion rank for actual champion (LSU), average CWS-participation
probability assigned to the 8 actual CWS teams, Brier score over CWS-binary outcome.

In [15]:
# Fit 3 Elo variants on cached 21-25 only (the cached file already cuts off at 2025-05-25)
g25 = (hist[hist['game_result'].isin(['W','L'])]
       .copy()
       .pipe(lambda d: d[~((d['year']==2021) & (d['conference']=='Ivy League'))]))
g25['unique_id'] = (g25['home_team'].apply(_r_str) + '_' +
                    g25['away_team'].apply(_r_str) + '_' +
                    g25['home_score'].apply(_r_str) + '_' +
                    g25['away_score'].apply(_r_str) + '_' +
                    g25['doubleheader_game'].apply(_r_str) + '_' +
                    g25['season_id'].apply(_r_str))
g25 = g25.drop_duplicates('unique_id', keep='first').copy()
g25['home_team_win'] = (g25['home_score'].astype(float) > g25['away_score'].astype(float)).astype(int)
g25['Date'] = pd.to_datetime(g25['Date'], format='%m/%d/%Y', errors='coerce')
g25 = g25.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
g25['score_diff'] = (g25['home_score'].astype(float) - g25['away_score'].astype(float)).abs().astype(int)
print(f"21-25 games for retrocast: {len(g25):,}  (cuts off {g25['Date'].max().date()})")

teams_25 = sorted(set(g25['home_team']) | set(g25['away_team']))
starting_25 = {t: 1500.0 for t in teams_25}

def fit_elo_pergame(games, starting, k_vec, hfa, r):
    ratings = dict(starting); starting_for_regress = dict(starting)
    n = len(games)
    probs = np.empty(n); actuals = games['home_team_win'].to_numpy()
    home, away = games['home_team'].to_numpy(), games['away_team'].to_numpy()
    neutral, year = games['neutral_site'].to_numpy(), games['year'].to_numpy()
    k_arr = np.full(n, k_vec, dtype=float) if np.isscalar(k_vec) else np.asarray(k_vec, dtype=float)
    prev_year = year[0]
    for i in range(n):
        if year[i] != prev_year:
            for t in ratings:
                ratings[t] += r * (starting_for_regress.get(t, 1500.0) - ratings[t])
            prev_year = year[i]
        h, a = home[i], away[i]
        rh = ratings.setdefault(h, 1500.0); ra = ratings.setdefault(a, 1500.0)
        p = expected_win_prob(rh, ra, hfa, neutral[i]); probs[i] = p
        ki = k_arr[i]
        ratings[h] = rh + ki * (actuals[i] - p)
        ratings[a] = ra + ki * ((1 - actuals[i]) - (1 - p))
    return EloFitResult(final_ratings=ratings, pregame_probs=probs, actuals=actuals)

# Baseline
fit_baseline = fit_elo_pergame(g25, starting_25, 29.0, 40.0, 0.07)

# MOV-weighted: walk baseline trajectory to get pre-game elos, derive per-game k
pre_h = np.empty(len(g25)); pre_a = np.empty(len(g25))
rw = {t:1500.0 for t in teams_25}; prev_yr = g25['year'].iat[0]
years = g25['year'].to_numpy(); ht = g25['home_team'].to_numpy(); at = g25['away_team'].to_numpy()
ns = g25['neutral_site'].to_numpy(); ac = g25['home_team_win'].to_numpy()
for i in range(len(g25)):
    if years[i] != prev_yr:
        for t in rw: rw[t] += 0.07*(1500.0 - rw[t])
        prev_yr = years[i]
    pre_h[i] = rw.setdefault(ht[i], 1500.0); pre_a[i] = rw.setdefault(at[i], 1500.0)
    p = expected_win_prob(pre_h[i], pre_a[i], 40.0, ns[i])
    rw[ht[i]] = pre_h[i] + 29 * (ac[i] - p)
    rw[at[i]] = pre_a[i] + 29 * ((1-ac[i]) - (1-p))
winner_elo = np.where(ac==1, pre_h, pre_a); loser_elo = np.where(ac==1, pre_a, pre_h)
winner_adv = winner_elo - loser_elo; sd = g25['score_diff'].to_numpy()
mov_mult = np.log(sd + 1) * 2.2 / (winner_adv * 0.001 + 2.2)
mov_mult = np.where(np.isfinite(mov_mult) & (mov_mult > 0), mov_mult, 1.0)
fit_mov = fit_elo_pergame(g25, starting_25, 29.0 * mov_mult, 40.0, 0.07)

# Recency-weighted: 60-day half-life
ref_date = g25['Date'].max()
days_before = (ref_date - g25['Date']).dt.days.to_numpy()
k_recency = np.maximum(29.0 * np.exp(-days_before / 60.0), 1.0)
fit_recency = fit_elo_pergame(g25, starting_25, k_recency, 40.0, 0.07)

print(f"\nFit diagnostics on 21-25:")
for name, fit in [('baseline',fit_baseline),('MOV',fit_mov),('recency-60d',fit_recency)]:
    print(f"  {name:12s}  MSE={fit.mse:.4f}  AUC={fit.auc:.4f}")

21-25 games for retrocast: 43,244  (cuts off 2025-05-25)



Fit diagnostics on 21-25:
  baseline      MSE=0.2149  AUC=0.6975
  MOV           MSE=0.2150  AUC=0.7027
  recency-60d   MSE=0.2347  AUC=0.6579


In [16]:
REGIONALS_2025 = {
    'nashville':    dict(seed=1, teams=['Vanderbilt','Wright St.','Louisville','ETSU']),
    'hattiesburg':  dict(seed=2, teams=['Southern Miss.','Columbia','Miami (FL)','Alabama']),
    'tallahassee':  dict(seed=3, teams=['Florida St.','Bethune-Cookman','Mississippi St.','Northeastern']),
    'corvallis':    dict(seed=4, teams=['Oregon St.',"Mount St. Mary's",'Southern California','TCU']),
    'chapel_hill':  dict(seed=5, teams=['North Carolina','Holy Cross','Nebraska','Oklahoma']),
    'eugene':       dict(seed=6, teams=['Oregon','Utah Valley','Cal Poly','Arizona']),
    'conway':       dict(seed=7, teams=['Coastal Carolina','Fairfield','East Carolina','Florida']),
    'auburn':       dict(seed=8, teams=['Auburn','Central Conn. St.','Stetson','NC State']),
    'austin':       dict(seed=9, teams=['Texas','Houston Christian','Kansas St.','UTSA']),
    'los_angeles':  dict(seed=10,teams=['UCLA','Fresno St.','UC Irvine','Arizona St.']),
    'oxford':       dict(seed=11,teams=['Ole Miss','Murray St.','Georgia Tech','Western Ky.']),
    'athens':       dict(seed=12,teams=['Georgia','Binghamton','Duke','Oklahoma St.']),
    'baton_rouge':  dict(seed=13,teams=['LSU','Little Rock','DBU','Rhode Island']),
    'clemson':      dict(seed=14,teams=['Clemson','USC Upstate','West Virginia','Kentucky']),
    'knoxville':    dict(seed=15,teams=['Tennessee','Miami (OH)','Wake Forest','Cincinnati']),
    'fayetteville': dict(seed=16,teams=['Arkansas','North Dakota St.','Creighton','Kansas']),
}
SEED2R_25 = {r['seed']: nm for nm, r in REGIONALS_2025.items()}
SUPER_PAIRS_25 = [(SEED2R_25[i], SEED2R_25[17-i]) for i in range(1,9)]

ACTUAL_CWS_25 = {'Coastal Carolina','Arizona','Oregon St.','Louisville','UCLA','Murray St.','Arkansas','LSU'}
ACTUAL_CHAMP_25 = 'LSU'

def _sim_game(t1, t2, ratings, rng):
    p = 1.0/(1.0+10.0**((ratings.get(t2,1500)-ratings.get(t1,1500))/400.0))
    return t1 if rng.random() < p else t2
def _sim_regional(teams, ratings, rng):
    g1=_sim_game(teams[0],teams[3],ratings,rng); g2=_sim_game(teams[1],teams[2],ratings,rng)
    g1l=teams[3] if g1==teams[0] else teams[0]; g2l=teams[2] if g2==teams[1] else teams[1]
    wbf=_sim_game(g1,g2,ratings,rng); wbfl=g2 if wbf==g1 else g1
    lg1=_sim_game(g1l,g2l,ratings,rng); lbf=_sim_game(wbfl,lg1,ratings,rng)
    f1=_sim_game(wbf,lbf,ratings,rng)
    return _sim_game(wbf,lbf,ratings,rng) if f1==lbf else f1
def _sim_tourney(ratings, rng):
    rwn={nm:_sim_regional(r['teams'],ratings,rng) for nm,r in REGIONALS_2025.items()}
    cws=[_sim_game(rwn[a],rwn[b],ratings,rng) for a,b in SUPER_PAIRS_25]
    b1=_sim_regional(cws[:4],ratings,rng); b2=_sim_regional(cws[4:],ratings,rng)
    return rwn, set(cws), _sim_game(b1,b2,ratings,rng)

def _evaluate(name, fit, n_sims=10000):
    ratings = dict(fit.final_ratings)
    all_t = {t for r in REGIONALS_2025.values() for t in r['teams']}
    for m in all_t - set(ratings): ratings[m] = 1500.0
    rng = np.random.default_rng(42)
    champ_ct = defaultdict(int); cws_ct = defaultdict(int)
    for _ in range(n_sims):
        _, cws, ch = _sim_tourney(ratings, rng)
        champ_ct[ch] += 1
        for t in cws: cws_ct[t] += 1
    champ_sorted = sorted(champ_ct.items(), key=lambda kv:-kv[1])
    cws_probs_actual = [cws_ct[t]/n_sims*100 for t in ACTUAL_CWS_25]
    return dict(name=name,
                cws_avg=float(np.mean(cws_probs_actual)),
                champ_pct_LSU=champ_ct[ACTUAL_CHAMP_25]/n_sims*100,
                champ_rank_LSU=next((i for i,(t,_) in enumerate(champ_sorted,1) if t==ACTUAL_CHAMP_25), -1),
                brier_cws=float(np.mean([(cws_ct[t]/n_sims - (1 if t in ACTUAL_CWS_25 else 0))**2 for t in ratings])),
                top5=[(t, champ_ct[t]/n_sims*100) for t,_ in champ_sorted[:5]])

rows = [_evaluate(n, f) for n, f in
        [('baseline', fit_baseline), ('MOV', fit_mov), ('recency-60d', fit_recency)]]
comp = pd.DataFrame(rows)[['name','cws_avg','champ_rank_LSU','champ_pct_LSU','brier_cws']]
comp.columns = ['variant','avg CWS-prob of actual 8 (%)','LSU champ rank','LSU champ %','Brier (CWS)']
print(comp.to_string(index=False))

print('\nTop-5 predicted champion per variant:')
for r in rows:
    print(f"  {r['name']:12s}: " + ', '.join(f"{t}({p:.1f}%)" for t,p in r['top5']))
comp.to_csv(PROJECT_ROOT / 'retrocast_2025_comparison.csv', index=False)
print("\nWrote retrocast_2025_comparison.csv")

    variant  avg CWS-prob of actual 8 (%)  LSU champ rank  LSU champ %  Brier (CWS)
   baseline                       21.4225               5         5.83     0.021568
        MOV                       20.5825               6         5.08     0.022919
recency-60d                       17.5450              27         0.92     0.021592

Top-5 predicted champion per variant:
  baseline    : Vanderbilt(16.3%), North Carolina(13.5%), Coastal Carolina(9.0%), Tennessee(7.6%), LSU(5.8%)
  MOV         : Vanderbilt(19.7%), North Carolina(17.9%), Coastal Carolina(10.2%), Northeastern(9.4%), Ole Miss(6.7%)
  recency-60d : Coastal Carolina(12.6%), Northeastern(12.0%), North Carolina(6.5%), Creighton(5.5%), Vanderbilt(5.4%)

Wrote retrocast_2025_comparison.csv


### Retrocast verdict

On the 2025 bracket — which featured unusual upsets (Louisville over Vandy, Murray St. over
Ole Miss, Cal Poly over Oregon) — none of the weighted variants clearly beat baseline:

- **MOV-weighted** sharpens confidence in top teams (Vanderbilt 19.7% vs baseline 16.3%) but
  Vanderbilt didn't even reach the CWS, so the extra confidence hurts.
- **Recency-60d** correctly elevates late-hot Coastal Carolina (1st pick) and Murray State
  (17.7% CWS-prob vs 4% baseline) but catastrophically deflates LSU (rank 27, 0.9%) since LSU
  was mid-season-injured.

**Decision: stick with the baseline 2025 model for the 2026 simulation.** No weighting applied.

Future iteration ideas (not currently implemented):
- Recency with a longer half-life (90–120 days) to be less aggressive
- MOV with a different multiplier formula or cap
- Combine MOV + light recency rather than either alone
- Pitcher-aware game probability (the biggest single accuracy win for college baseball)

## 9. Save artifacts

In [17]:
champ_probs.to_csv(PROJECT_ROOT / 'champ_probs_2026_py.csv', index=False)
cws_probs.to_csv(PROJECT_ROOT / 'cws_probs_2026_py.csv', index=False)
team_ranks.to_csv(PROJECT_ROOT / 'final_elo_ratings_2026_py.csv', index=False)
print('Wrote:')
print('  champ_probs_2026_py.csv')
print('  cws_probs_2026_py.csv')
print('  final_elo_ratings_2026_py.csv')

Wrote:
  champ_probs_2026_py.csv
  cws_probs_2026_py.csv
  final_elo_ratings_2026_py.csv


## 10. Next steps

Once this baseline is locked in:

1. **2025 retrocast.** Re-fit Elo on cached 2021–2025 data (which already cuts off at Selection Sunday 2025), run sims on the 2025 bracket, compare predictions to actual 2025 tournament results (Coastal Carolina won).
2. **Weighting bake-off.** Add MOV-weighted and recency-weighted variants, re-run retrocast for each, score by some combo of regional-winner accuracy + champion likelihood given actual.
3. **Apply winners to 2026.** If a weighting scheme improves the 2025 retrocast, ship it for 2026; otherwise keep baseline.